# LoRA rank study: run everything

Runtime > Change runtime type > **A100** (or L4). Run the cells in order.
Results are kept on Google Drive, so if the session disconnects, re-run all
cells and the study resumes where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
STORE = '/content/drive/MyDrive/lora-code-study'   # runs and results persist here

In [ ]:
REPO = 'https://github.com/SummmerKing/lora-code-study.git'
!test -d /content/lora-code-study || git clone $REPO /content/lora-code-study
%cd /content/lora-code-study
!git pull -q
!pip install -q -e ".[dev]"
# Keep outputs on Drive via symlinks.
!mkdir -p $STORE/runs $STORE/results $STORE/data
!rm -rf runs results data && ln -s $STORE/runs runs && ln -s $STORE/results results && ln -s $STORE/data data

## 1. Verify the code (CPU, under a minute)
Every check of the maths (masking, LoRA gradients, merging, statistics, sandbox) must pass before spending GPU time.

In [ ]:
!pytest -q

## 2. Data: decontaminate and tokenize

In [ ]:
!python scripts/prepare_data.py

## 3. Learning-rate sweep (selected by validation loss only)

In [ ]:
!python scripts/sweep_lr.py

## 4. Base model baseline

In [ ]:
!test -f results/base/summary.json || python scripts/evaluate.py --base

## 5. Train and evaluate 4 configurations x 3 seeds

In [ ]:
import subprocess, pathlib
for cfg in ['lora_r4', 'lora_r16', 'lora_r64', 'loraplus_r16']:
    for seed in [0, 1, 2]:
        run = pathlib.Path(f'runs/{cfg}/seed{seed}')
        if not (run / 'adapter.pt').exists():
            subprocess.run(['python', 'scripts/train.py', '--config', cfg, '--seed', str(seed)], check=True)
        if not pathlib.Path(f'results/{cfg}/seed{seed}/summary.json').exists():
            subprocess.run(['python', 'scripts/evaluate.py', '--run', str(run)], check=True)

## 6. Aggregate

In [ ]:
!python scripts/aggregate.py
from IPython.display import Image, display, Markdown
display(Markdown(open('results/summary.md').read()))
display(Image('results/figures/gain_over_base.png'))
display(Image('results/figures/val_loss.png'))

## 7. Read some failures yourself
Numbers are not enough: look at what the model actually got wrong.

In [ ]:
import json, random
rows = [json.loads(l) for l in open('results/lora_r16/seed0/humaneval.jsonl')]
fails = [r for r in rows if not r['passed']]
print(f"{len(fails)} failures")
for r in random.Random(0).sample(fails, min(3, len(fails))):
    print('=' * 80, r['task_id'], r['status'])
    print(r['response'][:1200])
    print('--- stderr:', r['stderr_tail'][-300:])